# WOS Coding Fine-Tuning on Colab

This notebook adapts the original public-dataset QLoRA SFT recipe to Google Colab for the **general coding assistant** model. It keeps the same overall causal-LM chat SFT method because that method absolutely can work for coding, but it now targets higher-end families that we can realistically access on Hugging Face without waiting on slow gated approvals.

## Candidate bases in this notebook

All three candidates are from **different model families**. Auto-selection exists only for one-off compatibility checks. It is **not** the final assignment workflow.

1. `google/gemma-2-27b-it` - strongest practical coding candidate here when you have Gemma access and enough VRAM.
2. `mistralai/Mistral-Small-3.1-24B-Instruct-2503` - strong ungated second-family option that stays easy to run.
3. `microsoft/phi-4` - ungated Phi-family replacement for the former Llama slot, with strong reasoning and coding capability for a 14B model.

To satisfy the assignment, you must complete **all three coding runs**: `gemma_2_27b`, `mistral_small_24b`, and `phi_4`. Together with the meeting notebook, that means **six total fine-tuning runs**.

## Before you run

1. Accept the Google license for Gemma at `https://huggingface.co/google/gemma-2-27b-it`
2. Mistral Small and Phi-4 do not require the same gated approval flow as the removed Llama candidate
3. Create a Colab secret named `HF_TOKEN` if you want the gated Gemma variant

## Audit of the original training method

- **Yes, it will work** for a coding model: instruction/chat SFT on a causal LM is a standard way to improve code completion, debugging, and explanation behavior.
- The original script actually makes **more sense for coding than for meetings**, especially because the examples are shorter and `packing = True` is less dangerous.
- The main limitations are not that it is invalid, but that it is **simple**: no masked-completion loss, no benchmark-aware curriculum, and no preference optimization.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import random
import re
import subprocess

try:
    drive = importlib.import_module("google.colab.drive")
except ModuleNotFoundError as exc:
    raise SystemExit("This notebook must run inside Google Colab.") from exc

drive.mount("/content/drive", force_remount=False)

WOS_DATA_ROOT = Path("/content/drive/MyDrive/wos_data/coding")
DATA_DIR = WOS_DATA_ROOT / "datasets"
RUNS_DIR = WOS_DATA_ROOT / "runs"
ARTIFACTS_DIR = WOS_DATA_ROOT / "artifacts"

for path in (DATA_DIR, RUNS_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "wos_data_root": str(WOS_DATA_ROOT),
    "data_dir": str(DATA_DIR),
    "runs_dir": str(RUNS_DIR),
    "artifacts_dir": str(ARTIFACTS_DIR),
}, indent=2))

In [ ]:
CODING_MODEL_OPTIONS = {
    "gemma_2_27b": {
        "model_name": "google/gemma-2-27b-it",
        "family": "Gemma",
        "gated": True,
        "min_vram_gib": 28.0,
        "max_seq_length": 2048,
        "learning_rate": 1e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/google/gemma-2-27b-it",
        "note": "Strongest practical coding candidate here when you have Gemma access and enough VRAM.",
    },
    "mistral_small_24b": {
        "model_name": "mistralai/Mistral-Small-3.1-24B-Instruct-2503",
        "family": "Mistral",
        "gated": False,
        "min_vram_gib": 28.0,
        "max_seq_length": 2048,
        "learning_rate": 1e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503",
        "note": "Strong ungated second-family option that stays easy to run.",
    },
    "phi_4": {
        "model_name": "microsoft/phi-4",
        "family": "Phi",
        "gated": False,
        "min_vram_gib": 18.0,
        "max_seq_length": 2048,
        "learning_rate": 1e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/microsoft/phi-4",
        "note": "Ungated Phi-family replacement for the former Llama slot, with strong reasoning and coding capability for a 14B model.",
    },
}

REQUIRED_MODEL_KEYS = ["gemma_2_27b", "mistral_small_24b", "phi_4"]
BASE_MODEL_KEY = REQUIRED_MODEL_KEYS[0]  # Change this for each of the 3 required coding runs. Use "auto" only for quick compatibility checks.

def detect_gpu():
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            text=True,
        ).strip()
    except Exception as exc:
        raise SystemExit("A GPU Colab runtime is required before training.") from exc

    rows = []
    for line in raw.splitlines():
        if not line.strip():
            continue
        name, memory = [part.strip() for part in line.split(",", 1)]
        mib = float(re.sub(r"[^0-9.]", "", memory))
        rows.append({
            "name": name,
            "memory_gib": round(mib / 1024.0, 2),
        })
    if not rows:
        raise SystemExit("No GPU was reported by nvidia-smi.")
    return rows[0]

def choose_model_key(requested_key: str, gpu_info: dict) -> str:
    if requested_key != "auto":
        if requested_key not in CODING_MODEL_OPTIONS:
            raise KeyError(f"Unknown BASE_MODEL_KEY: {requested_key}")
        return requested_key

    for key, option in CODING_MODEL_OPTIONS.items():
        if gpu_info["memory_gib"] >= option["min_vram_gib"]:
            return key
    raise SystemExit(
        f"No configured coding model fits this runtime. GPU memory: {gpu_info['memory_gib']:.1f} GiB."
    )

gpu = detect_gpu()
SELECTED_MODEL_KEY = choose_model_key(BASE_MODEL_KEY, gpu)
profile = CODING_MODEL_OPTIONS[SELECTED_MODEL_KEY]

print(json.dumps({
    "gpu": gpu,
    "requested_model_key": BASE_MODEL_KEY,
    "selected_model_key": SELECTED_MODEL_KEY,
    "required_model_keys": REQUIRED_MODEL_KEYS,
    "assignment_total_model_runs": 6,
    "available_models": CODING_MODEL_OPTIONS,
    "profile": profile,
}, indent=2))

In [ ]:
get_ipython().run_line_magic("pip", "install --upgrade pip")
get_ipython().run_line_magic("pip", "install \"accelerate>=1.4.0,<2.0.0\" \"bitsandbytes==0.49.2\" \"datasets>=4.8.0,<5.0.0\" \"huggingface-hub>=0.36.0,<1.0.0\" \"peft>=0.17.0,<1.0.0\" \"safetensors>=0.7.0\" \"sentencepiece>=0.2.0\" \"transformers>=4.57.0,<4.59.0\" \"trl==0.24.0\"")

import importlib.metadata as md
for package in ["transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes"]:
    print(f"{package}: {md.version(package)}")

In [ ]:
from huggingface_hub import HfApi
from jinja2.exceptions import TemplateError
from transformers import AutoTokenizer

HF_TOKEN = None
try:
    userdata = importlib.import_module("google.colab.userdata")
except ModuleNotFoundError:
    userdata = None

if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if profile.get("gated") and not HF_TOKEN:
    raise SystemExit(
        f"Set a Colab secret named HF_TOKEN before continuing with {profile['model_name']}. This model is gated."
    )
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi()
model_info_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
try:
    api.model_info(profile["model_name"], **model_info_kwargs)
except Exception as exc:
    if profile.get("gated"):
        raise RuntimeError(
            f"No access to {profile['model_name']}. Accept the license at {profile['access_url']} and rerun this cell. Original error: {exc}"
        ) from exc
    raise RuntimeError(
        f"Unable to reach {profile['model_name']} at {profile['access_url']}. Original error: {exc}"
    ) from exc

tokenizer = AutoTokenizer.from_pretrained(
    profile["model_name"],
    token=HF_TOKEN or None,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

template = tokenizer.chat_template or ""
if not template.strip():
    raise RuntimeError(f"{profile['model_name']} does not expose a usable chat template.")
if tokenizer.eos_token_id is None:
    raise RuntimeError(f"{profile['model_name']} does not expose an eos_token_id.")

def fold_system_message_into_user(messages: list[dict]) -> list[dict]:
    system_chunks = [
        message["content"].strip()
        for message in messages
        if message["role"] == "system" and message.get("content")
    ]
    remaining = [dict(message) for message in messages if message["role"] != "system"]
    if not system_chunks:
        return [dict(message) for message in messages]
    merged_system = "\n\n".join(chunk for chunk in system_chunks if chunk).strip()
    if not remaining:
        return [{"role": "user", "content": merged_system}]
    first_message = dict(remaining[0])
    if first_message["role"] == "user":
        user_content = first_message.get("content", "")
        first_message["content"] = f"{merged_system}\n\n{user_content}".strip()
        remaining[0] = first_message
    else:
        remaining.insert(0, {"role": "user", "content": merged_system})
    return remaining

def apply_chat_template_safe(active_tokenizer, messages, **kwargs):
    try:
        return active_tokenizer.apply_chat_template(messages, **kwargs)
    except TemplateError as exc:
        if "System role not supported" not in str(exc):
            raise
        return active_tokenizer.apply_chat_template(
            fold_system_message_into_user(messages),
            **kwargs,
        )

sample_messages = [
    {"role": "system", "content": "You are a careful coding assistant."},
    {"role": "user", "content": "Write a Python function that returns the two numbers that sum to the target."},
    {"role": "assistant", "content": "def two_sum(nums, target): ..."},
]
rendered = apply_chat_template_safe(tokenizer, sample_messages, tokenize=False, add_generation_prompt=False)
generation_prompt = apply_chat_template_safe(tokenizer, sample_messages[:2], tokenize=False, add_generation_prompt=True)
if not rendered.strip() or not generation_prompt.strip():
    raise RuntimeError(f"Chat template rendering failed for {profile['model_name']}.")
if sample_messages[1]["content"] not in rendered:
    raise RuntimeError(f"Rendered prompt for {profile['model_name']} did not preserve the user turn.")

print(json.dumps({
    "model_name": profile["model_name"],
    "family": profile["family"],
    "selected_model_key": SELECTED_MODEL_KEY,
    "gated": profile.get("gated", False),
    "eos_token": tokenizer.eos_token,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token": tokenizer.pad_token,
    "pad_token_id": tokenizer.pad_token_id,
    "chat_template_present": True,
    "access_url": profile["access_url"],
    "rendered_preview": rendered[:700],
    "generation_preview": generation_prompt[:700],
}, indent=2))

In [ ]:
import hashlib
from collections import Counter

from datasets import load_dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

SYSTEM_PROMPT = (
    "You are WOS Coding, an expert software engineer assistant. "
    "You write clean, correct, well-structured code, explain technical concepts clearly, "
    "debug issues systematically, and follow best practices for the requested language."
)

TRAIN_JSONL = DATA_DIR / "train.jsonl"
TRAIN_SPLIT_JSONL = DATA_DIR / "train_split.jsonl"
EVAL_SPLIT_JSONL = DATA_DIR / "eval_split.jsonl"
DATASET_MANIFEST = DATA_DIR / "manifest.json"

def to_record(prompt: str, response: str, source: str) -> dict:
    return {
        "source": source,
        "conversations": [
            {"from": "system", "value": SYSTEM_PROMPT},
            {"from": "human", "value": prompt},
            {"from": "gpt", "value": response},
        ],
    }

def download_codefeedback(n: int) -> list[dict]:
    print("Downloading CodeFeedback-Filtered-Instruction...")
    ds = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train")
    samples = []
    for row in tqdm(ds, desc="CodeFeedback"):
        query = row.get("query", "")
        answer = row.get("answer", "")
        if not query or not answer or len(query) < 10 or len(answer) < 20:
            continue
        samples.append(to_record(query, answer, "codefeedback"))
        if len(samples) >= n:
            break
    return samples

def download_codealpaca(n: int) -> list[dict]:
    print("Downloading CodeAlpaca-20k...")
    ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    samples = []
    for row in tqdm(ds, desc="CodeAlpaca"):
        instruction = row.get("instruction", "")
        inp = row.get("input", "")
        output = row.get("output", "")
        if not instruction or not output:
            continue
        prompt = f"{instruction}\n\n{inp}".strip() if inp else instruction
        samples.append(to_record(prompt, output, "codealpaca"))
        if len(samples) >= n:
            break
    return samples

def download_python_instructions(n: int) -> list[dict]:
    print("Downloading Python Instructions dataset...")
    try:
        ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
    except Exception as exc:
        print(f"Python instructions load failed: {exc} -- skipping")
        return []
    samples = []
    for row in tqdm(ds, desc="Python Instructions"):
        instruction = row.get("instruction", "")
        inp = row.get("input", "")
        output = row.get("output", "")
        if not instruction or not output:
            continue
        prompt = f"{instruction}\n\n{inp}".strip() if inp else instruction
        samples.append(to_record(prompt, output, "python_instructions"))
        if len(samples) >= n:
            break
    return samples

codefeedback = download_codefeedback(40000)
codealpaca = download_codealpaca(12000)
python_inst = download_python_instructions(8000)
all_samples = codefeedback + codealpaca + python_inst
random.shuffle(all_samples)
all_samples = all_samples[:60000]

split_index = int(len(all_samples) * 0.95)
train_split = all_samples[:split_index]
eval_split = all_samples[split_index:]

for target_path, records in [
    (TRAIN_JSONL, all_samples),
    (TRAIN_SPLIT_JSONL, train_split),
    (EVAL_SPLIT_JSONL, eval_split),
]:
    with target_path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

manifest = {
    "seed": SEED,
    "total_samples": len(all_samples),
    "train_samples": len(train_split),
    "eval_samples": len(eval_split),
    "source_counts": dict(Counter(record["source"] for record in all_samples)),
    "train_jsonl": str(TRAIN_JSONL),
    "train_split_jsonl": str(TRAIN_SPLIT_JSONL),
    "eval_split_jsonl": str(EVAL_SPLIT_JSONL),
}
DATASET_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def row_hash(row: dict) -> str:
    payload = json.dumps(row["conversations"], sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

train_rows = read_jsonl(TRAIN_SPLIT_JSONL)
eval_rows = read_jsonl(EVAL_SPLIT_JSONL)
train_hashes = {row_hash(row) for row in train_rows}
eval_hashes = {row_hash(row) for row in eval_rows}

def prompt_length(row: dict) -> int:
    return len(row["conversations"][1]["value"])

def response_length(row: dict) -> int:
    return len(row["conversations"][2]["value"])

summary = {
    "exact_train_eval_overlap": len(train_hashes & eval_hashes),
    "train_source_counts": dict(Counter(row["source"] for row in train_rows)),
    "eval_source_counts": dict(Counter(row["source"] for row in eval_rows)),
    "max_train_prompt_chars": max(prompt_length(row) for row in train_rows),
    "max_eval_prompt_chars": max(prompt_length(row) for row in eval_rows),
    "max_train_response_chars": max(response_length(row) for row in train_rows),
    "max_eval_response_chars": max(response_length(row) for row in eval_rows),
    "sample_prompt_preview": train_rows[0]["conversations"][1]["value"][:700],
    "sample_response_preview": train_rows[0]["conversations"][2]["value"][:500],
}
print(json.dumps(summary, indent=2))
if summary["exact_train_eval_overlap"]:
    raise RuntimeError("Exact duplicate leakage detected between train and eval splits.")

In [ ]:
START_TRAINING_NOW = False
MERGE_AFTER_TRAIN = False
PUSH_ADAPTER_TO_HUB = False
HF_MODEL_REPO = None  # example: your-handle/wos-coding-llama-3-3-70b-lora

MAX_TRAIN_SAMPLES = 6000
NUM_TRAIN_EPOCHS = 1.0
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
PACKING = True
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2
REPORT_TO = "none"

OUTPUT_DIR = RUNS_DIR / f"wos-coding-{SELECTED_MODEL_KEY}-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
from datasets import Dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_jsonl_dataset(path: Path) -> Dataset:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

def format_record(example: dict) -> dict:
    role_map = {"system": "system", "human": "user", "gpt": "assistant"}
    messages = [
        {"role": role_map[turn["from"]], "content": turn["value"]}
        for turn in example["conversations"]
    ]
    return {
        "text": apply_chat_template_safe(tokenizer, messages, tokenize=False, add_generation_prompt=False)
    }

def infer_lora_target_modules(model) -> list[str]:
    common_suffixes = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "qkv_proj", "gate_up_proj", "gate_proj", "up_proj", "down_proj",
        "query_key_value", "c_attn", "c_proj", "dense", "fc1", "fc2",
    ]
    matches = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        suffix = name.split(".")[-1]
        if suffix in common_suffixes:
            matches.append(suffix)
    ordered = [suffix for suffix in common_suffixes if suffix in matches]
    if not ordered:
        raise RuntimeError("Unable to infer LoRA target modules for this architecture.")
    return ordered

def find_latest_checkpoint(path: Path):
    checkpoints = []
    for candidate in path.glob("checkpoint-*"):
        if candidate.is_dir():
            try:
                step = int(candidate.name.split("-", 1)[1])
            except (IndexError, ValueError):
                continue
            checkpoints.append((step, candidate))
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda item: item[0])
    return checkpoints[-1][1]

def choose_training_dtype() -> torch.dtype:
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

training_plan = {
    "requested_model_key": BASE_MODEL_KEY,
    "selected_model_key": SELECTED_MODEL_KEY,
    "model_name": profile["model_name"],
    "access_url": profile["access_url"],
    "output_dir": str(OUTPUT_DIR),
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": profile["learning_rate"],
    "max_seq_length": profile["max_seq_length"],
    "gradient_accumulation_steps": profile["gradient_accumulation_steps"],
    "packing": PACKING,
    "push_adapter_to_hub": PUSH_ADAPTER_TO_HUB,
    "merge_after_train": MERGE_AFTER_TRAIN,
    "resume_from_checkpoint": str(find_latest_checkpoint(OUTPUT_DIR)) if find_latest_checkpoint(OUTPUT_DIR) else None,
}
print(json.dumps(training_plan, indent=2))

if START_TRAINING_NOW:
    train_dataset = load_jsonl_dataset(TRAIN_SPLIT_JSONL)
    eval_dataset = load_jsonl_dataset(EVAL_SPLIT_JSONL)
    if MAX_TRAIN_SAMPLES and len(train_dataset) > MAX_TRAIN_SAMPLES:
        train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))

    train_dataset = train_dataset.map(format_record, remove_columns=train_dataset.column_names)
    eval_dataset = eval_dataset.map(format_record, remove_columns=eval_dataset.column_names)

    compute_dtype = choose_training_dtype()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        profile["model_name"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        token=HF_TOKEN or None,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_targets = infer_lora_target_modules(model)
    print("LoRA target modules:", lora_targets)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        target_modules=lora_targets,
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=profile["gradient_accumulation_steps"],
        learning_rate=profile["learning_rate"],
        weight_decay=0.01,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        bf16=compute_dtype == torch.bfloat16,
        fp16=compute_dtype == torch.float16,
        optim="paged_adamw_8bit",
        seed=SEED,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        report_to=[] if REPORT_TO == "none" else [REPORT_TO],
        gradient_checkpointing=True,
        disable_tqdm=False,
        remove_unused_columns=False,
        dataset_text_field="text",
        max_length=profile["max_seq_length"],
        packing=PACKING,
        dataset_num_proc=1,
        logging_first_step=True,
        save_safetensors=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)
    train_result = trainer.train(
        resume_from_checkpoint=str(latest_checkpoint) if latest_checkpoint else None
    )
    print(json.dumps(train_result.metrics, indent=2, default=str))

    adapter_path = OUTPUT_DIR / "adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"Adapter saved to {adapter_path}")

    if PUSH_ADAPTER_TO_HUB:
        if not HF_MODEL_REPO or not HF_TOKEN:
            raise RuntimeError("Set HF_MODEL_REPO and HF_TOKEN before pushing adapters.")
        api = HfApi(token=HF_TOKEN or None)
        api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN or None)
        api.upload_folder(
            folder_path=str(adapter_path),
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            path_in_repo="adapter",
            commit_message="Add coding adapter",
            token=HF_TOKEN or None,
        )
        print(f"Adapter pushed to https://huggingface.co/{HF_MODEL_REPO}")

    if MERGE_AFTER_TRAIN:
        merged_path = OUTPUT_DIR / "merged"
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(merged_path, safe_serialization=True, max_shard_size="5GB")
        tokenizer.save_pretrained(merged_path)
        print(f"Merged model saved to {merged_path}")
else:
    print("Training not started. Set START_TRAINING_NOW = True and re-run this cell when ready.")

In [ ]:
RUN_SMOKE_TEST = False
SMOKE_TEST_SOURCE = "adapter"  # adapter | merged
SMOKE_TEST_PROMPT = "Write a Python function `two_sum(nums, target)` that returns the indices of the two numbers that add up to `target`. Return only code."

if RUN_SMOKE_TEST:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if SMOKE_TEST_SOURCE == "merged":
        model_path = OUTPUT_DIR / "merged"
        if not model_path.exists():
            raise FileNotFoundError("Merged model not found. Run training with MERGE_AFTER_TRAIN = True first.")
        smoke_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )
        smoke_tokenizer = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
    else:
        adapter_path = OUTPUT_DIR / "adapter"
        if not adapter_path.exists():
            raise FileNotFoundError("Adapter not found. Run the training cell first.")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        smoke_tokenizer = AutoTokenizer.from_pretrained(profile["model_name"], token=HF_TOKEN, trust_remote_code=True)
        if smoke_tokenizer.pad_token is None:
            smoke_tokenizer.pad_token = smoke_tokenizer.eos_token
        smoke_model = AutoModelForCausalLM.from_pretrained(
            profile["model_name"],
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            token=HF_TOKEN,
        )
        smoke_model = PeftModel.from_pretrained(smoke_model, str(adapter_path))

    smoke_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": SMOKE_TEST_PROMPT},
    ]
    inputs = apply_chat_template_safe(
        smoke_tokenizer,
        smoke_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(smoke_model.device)

    with torch.no_grad():
        outputs = smoke_model.generate(
            inputs,
            max_new_tokens=300,
            do_sample=False,
            eos_token_id=smoke_tokenizer.eos_token_id,
            pad_token_id=smoke_tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs.shape[1]:]
    print(smoke_tokenizer.decode(generated, skip_special_tokens=True))
else:
    print("Smoke test not run. Set RUN_SMOKE_TEST = True after adapter or merged weights exist.")

## Evaluation

This section gives you metrics that fit the way the coding model is trained.

- `mbpp_pass_at_1` is the strongest metric here: it checks whether the first generated solution passes benchmark unit tests.
- `SWE-bench` or `SWE-bench Verified` is stronger for repo-level software engineering, but it is much heavier and is better treated as a phase-2 benchmark than the fast Colab baseline here.
- `syntax_valid_rate` tells you how often the model produces valid Python.
- `code_extract_rate` tells you how often the assistant response can be cleanly turned into executable code.

Unlike meeting summarization, lexical overlap is a weak metric for code because many correct programs can look very different. Execution-based metrics make more sense for this chat-SFT setup, with MBPP as the quick baseline and SWE-bench as the stronger follow-up benchmark.

In [ ]:
RUN_BENCHMARK_EVAL = False
EVAL_SOURCE = "adapter"  # adapter | merged
EVAL_MAX_SAMPLES = 40
EXECUTION_TIMEOUT_SECONDS = 10
EVAL_ARTIFACT_PATH = ARTIFACTS_DIR / f"coding_eval_{SELECTED_MODEL_KEY}.jsonl"

import ast
import sys
import tempfile
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def extract_python_code(text: str) -> str:
    fenced_block = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if fenced_block:
        return fenced_block.group(1).strip()
    return text.strip()

def is_valid_python(code: str) -> bool:
    if not code.strip():
        return False
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

def load_coding_eval_model(eval_source: str):
    if eval_source == "merged":
        model_path = OUTPUT_DIR / "merged"
        if not model_path.exists():
            raise FileNotFoundError("Merged model not found. Run training with MERGE_AFTER_TRAIN = True first.")
        eval_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )
        eval_tokenizer = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
    else:
        adapter_path = OUTPUT_DIR / "adapter"
        if not adapter_path.exists():
            raise FileNotFoundError("Adapter not found. Run the training cell first.")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        eval_tokenizer = AutoTokenizer.from_pretrained(profile["model_name"], token=HF_TOKEN or None, trust_remote_code=True)
        if eval_tokenizer.pad_token is None:
            eval_tokenizer.pad_token = eval_tokenizer.eos_token
        base_model = AutoModelForCausalLM.from_pretrained(
            profile["model_name"],
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            token=HF_TOKEN or None,
        )
        eval_model = PeftModel.from_pretrained(base_model, str(adapter_path))
    return eval_model, eval_tokenizer

def generate_code_response(eval_model, eval_tokenizer, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    inputs = apply_chat_template_safe(
        eval_tokenizer,
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            inputs,
            max_new_tokens=400,
            do_sample=False,
            eos_token_id=eval_tokenizer.eos_token_id,
            pad_token_id=eval_tokenizer.pad_token_id,
        )
    generated = outputs[0][inputs.shape[1]:]
    return eval_tokenizer.decode(generated, skip_special_tokens=True).strip()

def load_mbpp_eval_rows(limit: int):
    try:
        benchmark = load_dataset("mbpp", "sanitized", split="test")
    except Exception:
        benchmark = load_dataset("mbpp", split="test")
    capped = min(limit, len(benchmark))
    return list(benchmark.select(range(capped)))

def run_python_tests(candidate_code: str, setup_code: str, tests: list[str], timeout_seconds: int):
    program_parts = [
        "import math",
        "import re",
        "import itertools",
        "import collections",
        setup_code or "",
        candidate_code,
        *tests,
    ]
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as handle:
        handle.write("\n\n".join(part for part in program_parts if part))
        temp_path = handle.name
    try:
        completed = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
        )
        return completed.returncode == 0, (completed.stderr or completed.stdout).strip()
    except subprocess.TimeoutExpired:
        return False, "timeout"
    finally:
        try:
            os.remove(temp_path)
        except OSError:
            pass

if RUN_BENCHMARK_EVAL:
    benchmark_rows = load_mbpp_eval_rows(EVAL_MAX_SAMPLES)
    eval_model, eval_tokenizer = load_coding_eval_model(EVAL_SOURCE)

    code_extract_count = 0
    syntax_valid_count = 0
    pass_count = 0
    results = []

    for row in tqdm(benchmark_rows, desc="Coding evaluation"):
        prompt = (
            "Write Python code that solves the following problem. Return only code.\n\n"
            f"{row['text']}"
        )
        raw_prediction = generate_code_response(eval_model, eval_tokenizer, prompt)
        code_prediction = extract_python_code(raw_prediction)
        extracted_code = bool(code_prediction)
        syntax_valid = extracted_code and is_valid_python(code_prediction)
        tests_passed = False
        execution_error = None

        if extracted_code:
            code_extract_count += 1
        if syntax_valid:
            syntax_valid_count += 1
            tests_passed, execution_error = run_python_tests(
                code_prediction,
                row.get("test_setup_code", ""),
                row.get("test_list", []),
                EXECUTION_TIMEOUT_SECONDS,
            )
        if tests_passed:
            pass_count += 1

        results.append({
            "task_id": row.get("task_id"),
            "prompt": row["text"],
            "reference_code": row.get("code"),
            "raw_prediction": raw_prediction,
            "code_prediction": code_prediction,
            "extracted_code": extracted_code,
            "syntax_valid": syntax_valid,
            "tests_passed": tests_passed,
            "execution_error": execution_error,
        })

    with EVAL_ARTIFACT_PATH.open("w", encoding="utf-8") as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")

    count = len(results)
    summary = {
        "model_name": profile["model_name"],
        "selected_model_key": SELECTED_MODEL_KEY,
        "eval_source": EVAL_SOURCE,
        "benchmark": "mbpp",
        "eval_samples": count,
        "code_extract_rate": code_extract_count / count if count else 0.0,
        "syntax_valid_rate": syntax_valid_count / count if count else 0.0,
        "mbpp_pass_at_1": pass_count / count if count else 0.0,
        "artifact_path": str(EVAL_ARTIFACT_PATH),
    }
    print(json.dumps(summary, indent=2))
else:
    print("Benchmark evaluation not run. Set RUN_BENCHMARK_EVAL = True after adapter or merged weights exist.")